# Data Ingestion with APIs and Webhooks
**Workshop, 120 minutes. No coding needed.** You only run cells (press the play button).
Some cells have one value you can change. Those are marked **Exercise**.

*Data ingestion* means getting data from somewhere else into your own place. Today we do it two ways: **pull** and **push**.

Run the cells from top to bottom.

**If something breaks:** use the menu **Runtime → Run before** (Colab runs every cell above the one you picked). Do not use *Run all*: it would place extra orders.

## 0. Setup
Type your name below. The presenter will tell you the real `CLASS_TOPIC`. Then run the cell.

In [ ]:
MY_NAME = "anon"  # @param {type:"string"}
CLASS_TOPIC = "ws-demo-class"  # @param {type:"string"}   (used in Part 5)

print("Hello", MY_NAME)

**Helpers.** Just run this cell. You do not need to read it.
It holds small functions we use later. It also holds some sample orders, used if the internet service is down.

In [ ]:
#@title Helpers - just run this cell { display-mode: "form" }
# Nothing to edit here. These small functions are used by later cells.
import json, os, time, uuid
from datetime import datetime, timezone
import requests
import pandas as pd

NTFY = "https://ntfy.sh"
SMEE = "https://smee.io"
# Only for testing: pretend ntfy and smee are down.
FORCE_FALLBACK = os.environ.get("FORCE_FALLBACK") == "1"

# Sample data used when ntfy is down or says "too many requests" (HTTP 429).
# Messages are STRINGS, just like real ntfy messages. Note: 2 duplicates, 1 bad message.
def _o(oid, who, pid, qty):
    return json.dumps({"order_id": oid, "customer": who, "product_id": pid,
                       "qty": qty, "ts": "2026-10-09T10:00:00+00:00"})
SAMPLE_ORDERS = [_o("a1", "Ana", 1, 2), _o("a2", "Budi", 11, 1), _o("a3", "Citra", 12, 3),
    _o("a4", "Dewi", 13, 1), _o("a5", "Eko", 14, 2), _o("a6", "Fajar", 15, 1),
    _o("a7", "Gita", 1, 1), _o("a8", "Hadi", 11, 4), _o("a9", "Indra", 12, 1),
    _o("b1", "Joko", 13, 2), _o("b2", "Kiki", 1, 5), _o("b3", "Lina", 14, 1),
    _o("a2", "Budi", 11, 1),            # duplicate of a2
    _o("b1", "Joko", 13, 2),            # duplicate of b1
    "oops this is not json {"]          # malformed message
SAMPLE_EVENTS = [{"id": f"s{i}", "event": "message", "message": m}
                 for i, m in enumerate(SAMPLE_ORDERS)]

def new_smee_channel():
    """Make a free smee.io receiver URL. Returns the URL, or None if smee is down."""
    try:
        if FORCE_FALLBACK:
            raise requests.ConnectionError("FORCE_FALLBACK is on")
        r = requests.get(f"{SMEE}/new", allow_redirects=False, timeout=10)
        url = r.headers["Location"]
        if not url.startswith(SMEE + "/"):
            raise ValueError("unexpected reply from smee")
        return url
    except Exception as e:
        print(f"WARNING: could not reach smee.io ({e}).")
        return None

def publish(topic, payload):
    """POST a dict as JSON text to ntfy. Returns the message id, or None on failure."""
    try:
        if FORCE_FALLBACK:
            raise requests.ConnectionError("FORCE_FALLBACK is on")
        r = requests.post(f"{NTFY}/{topic}", data=json.dumps(payload), timeout=10)
        r.raise_for_status()
        return r.json()["id"]
    except Exception as e:
        print(f"WARNING: could not publish to ntfy ({e}).")
        return None

def poll(topic, since="all"):
    """Ask ntfy for stored messages after `since`. Falls back to SAMPLE_EVENTS."""
    try:
        if FORCE_FALLBACK:
            raise requests.ConnectionError("FORCE_FALLBACK is on")
        r = requests.get(f"{NTFY}/{topic}/json", params={"poll": 1, "since": since}, timeout=10)
        r.raise_for_status()
        events = [json.loads(line) for line in r.text.splitlines() if line.strip()]
    except Exception as e:
        print(f"WARNING: ntfy not available ({e}). Using SAMPLE_ORDERS instead.")
        ids = [ev["id"] for ev in SAMPLE_EVENTS]
        start = ids.index(since) + 1 if since in ids else 0   # only events AFTER the cursor
        return SAMPLE_EVENTS[start:]
    return [e for e in events if e.get("event") == "message"]  # skip open/keepalive

def stream(topic, seconds, on_event):
    """Listen live for `seconds`, call on_event(event) for each real message."""
    end = time.time() + seconds
    try:
        if FORCE_FALLBACK:
            raise requests.ConnectionError("FORCE_FALLBACK is on")
        with requests.get(f"{NTFY}/{topic}/json", stream=True, timeout=(10, seconds)) as r:
            r.raise_for_status()
            try:
                for line in r.iter_lines():
                    if line and json.loads(line).get("event") == "message":
                        on_event(json.loads(line))
                    if time.time() > end:
                        break
            except requests.RequestException:
                pass  # quiet topic: read timed out, that is the normal end
    except Exception as e:
        print(f"WARNING: could not stream from ntfy ({e}).")


## 1. API (pull)
Think of a restaurant. You sit at a table. You ask the waiter for a dish. The kitchen answers. **You start every request.**

An **API** works the same way. Your code asks a server for data. The server replies.
We use DummyJSON, a free fake shop. No account needed.

In [ ]:
# The base URL is the front door of the shop service
BASE_URL = "https://dummyjson.com"

# Ask for product number 1
r = requests.get(BASE_URL + "/products/1", timeout=10)
data = r.json()
print("Status:", r.status_code)   # 200 means OK
print(data)

**Exercise:** change the number `1` to any number from 1 to 194, then run the cell.

In [ ]:
PRODUCT_ID = 1  # @param {type:"integer"}
r = requests.get(f"{BASE_URL}/products/{PRODUCT_ID}", timeout=10)
r.raise_for_status()   # stops with an error if there is no such product (404)
data = r.json()
print(r.status_code, data["title"], "- price:", data["price"])

Now we **search**. We ask for products that match the word `phone`.

In [ ]:
r = requests.get(BASE_URL + "/products/search", params={"q": "phone"}, timeout=10)
data = r.json()
print("Found:", data["total"])
for p in data["products"]:
    print("-", p["title"], p["price"])

The shop has 194 products. The server will not send them all at once. It sends small groups.
Each group is a page. This is called **pagination**.
`limit` = how many items per page. `skip` = how many items to jump over.

In [ ]:
# Page 3 if each page has 5 items: skip 10, take 5 (ids 11 to 15)
r = requests.get(BASE_URL + "/products", params={"skip": 10, "limit": 5}, timeout=10)
data = r.json()
print("ids:", [p["id"] for p in data["products"]], "| total:", data["total"])

Now a loop gets every page and puts all rows in a table (a pandas **DataFrame**).

In [ ]:
# Loop over every page and collect one row per product
rows, skip, total, limit = [], 0, 1, 50
while skip < total:
    r = requests.get(BASE_URL + "/products", params={"skip": skip, "limit": limit}, timeout=10)
    r.raise_for_status()
    data = r.json()
    total = data["total"]            # how many products exist in all
    for p in data["products"]:
        rows.append({"id": p["id"], "title": p["title"], "category": p["category"], "price": p["price"]})
    skip += limit                    # jump to the next page

products = pd.DataFrame(rows)
print(len(products), "products")
products.head()

Servers protect themselves. They limit how many requests you can send. This is a **rate limit**.
The server tells you what is left in a header. Please do not spam it.

In [ ]:
r = requests.get(BASE_URL + "/products/1", timeout=10)
print("Limit:", r.headers.get("x-ratelimit-limit"))
print("Remaining:", r.headers.get("x-ratelimit-remaining"))

## 2. Webhook / push
Now the opposite. "Don't call us, we'll call you."
Think of a pizza shop that rings your phone when the pizza is ready. **The sender starts the message.**

Three words:
- **Event**: something that happened (an order was placed).
- **Payload**: the data sent with the event (the order details).
- **Receiver URL**: the web address that gets the event.

A real shop (Midtrans, Shopify) sends a **webhook** to your receiver URL.
Today we use **smee.io**. It is a free tool that developers use to look at webhooks. Think of it as a **window**: it shows you exactly what arrives, but it keeps nothing. To be honest: smee is only a stand-in receiver. We use it so we need no account.

In [ ]:
# Make your own receiver URL
SMEE_URL = new_smee_channel()
if SMEE_URL:
    print("Open this link in a new tab, then keep it open:")
    print(SMEE_URL)
else:
    print("smee.io is not available right now. No problem: the next cells show a sample instead.")

In [ ]:
# Send one sample order as a webhook. Watch it appear in the other tab.
order = {"order_id": "demo1", "customer": MY_NAME, "product_id": 1, "qty": 2}
if SMEE_URL:
    try:
        r = requests.post(SMEE_URL, json=order, headers={"X-Source": "toko-kelas"}, timeout=10)
        print("Sent! Status", r.status_code, "(200 = smee received it). Look at the other tab.")
    except requests.RequestException as e:
        print("WARNING: could not send to smee:", e)
else:
    print("No smee link, so nothing was sent. A real webhook looks like this:")
    print("  Headers: content-type = application/json | x-source = toko-kelas")
    print("  Body:   ", json.dumps(order))

**Exercise:** change `3` if you like, then run. Watch the tab.

In [ ]:
HOW_MANY = 3  # @param {type:"integer"}
if SMEE_URL:
    try:
        for i in range(HOW_MANY):
            order = {"order_id": f"demo{i+2}", "customer": MY_NAME, "product_id": i + 1, "qty": 1}
            requests.post(SMEE_URL, json=order, headers={"X-Source": "toko-kelas"}, timeout=10)
        print("Sent", HOW_MANY, "more orders")
    except requests.RequestException as e:
        print("WARNING: could not send to smee:", e)
else:
    print("No smee link, so nothing was sent.")

**What did you see?** In the tab, each request shows two parts:
- **Headers**: labels about the message, like who sent it.
- **Body**: the data inside. This is the **payload**, your order.

That is what a real webhook looks like.
Smee is a **window**: it shows the webhook but keeps nothing. Close the tab and it is gone. In Part 5 we switch to a **mailbox** that does keep messages.

## 3. Pull vs push

| | Pull (API) | Push (webhook) |
|---|---|---|
| Who starts? | You ask | The sender sends |
| Freshness | As fresh as your last ask | Almost instant |
| Load | You choose how often | You must be ready any time |
| Missed data | Safe: ask again later | Can be lost if you are offline |
| History | Often kept, you can go back | Usually not kept for long |

**Quiz 1:** You need a report every morning. Pull or push?

<details><summary>Answer</summary>Pull. You choose the time and ask once.</details>

**Quiz 2:** Your site must react the second a payment is made. Pull or push?

<details><summary>Answer</summary>Push. The sender tells you right away. (A pull loop would be slow or wasteful.)</details>

## 4. Role in data engineering
A data pipeline has steps:

**Source → Ingest → Store → Transform → Serve**

**Ingest** is the step where data enters your system. That is today's topic.

- **API = pull.** Batch or incremental. You deal with pagination, cursors and rate limits.
- **Push = events.** Land the raw data first. Expect duplicates and gaps.
- **In production** teams combine both.

## 5. Hands-on mini pipeline
**We switch tools.** In Part 2 we used smee: a window that keeps nothing.
Now we want to collect, clean and re-run. For that we need a place that **keeps** the orders. So we use **ntfy**: a free mailbox that stores messages for a while.
(Both are stand-ins. A real company builds its own receiver and storage.)

Everyone sends orders to **one shared topic** (the mailbox). Then we collect, clean and analyse them together.

**How to run Part 5.** Each step says how often to run it:

| Step | What it does | How often |
|---|---|---|
| 5a Place order | Sends an order | As many times as you like |
| 5a Resend | Sends the same order again | Once |
| 5b Collect | **Saves** new orders in your notebook | Many times, after orders are placed |
| 5c Watch | Presenter demo. **Skip this step.** Saves nothing. | Presenter only |
| 5d Clean | Cleans what 5b saved | Once, after 5b |
| 5e Enrich | Adds product names and revenue | Once, after 5d |
| 5f CSV | Saves files (optional) | Once |

**Remember:** 5b *collects*. 5c is only a live demo by the presenter, so skip it. Run 5b again any time to collect newer orders.

### 5a. Producer
Pick a product (1 to 194) and a quantity. Run the cell to place an order.

In [ ]:
PRODUCT = 1  # @param {type:"integer"}
QTY = 1  # @param {type:"integer"}
last_order = {"order_id": uuid.uuid4().hex[:8], "customer": MY_NAME, "product_id": PRODUCT,
              "qty": QTY, "ts": datetime.now(timezone.utc).isoformat()}
publish(CLASS_TOPIC, last_order)
print("Placed:", last_order)

Oops, the network was slow, so you press send again. Senders often retry. The **same** order arrives **twice**. We call this a duplicate. Keep it in mind for 5d.

> **Run: once.** (Run the next cell one time only. That is your duplicate.)

In [ ]:
# Oops, resend my last order (same order_id!)
publish(CLASS_TOPIC, last_order)
print("Resent:", last_order["order_id"])

### 5b. Batch consumer
Now we read the orders. A broker can be a little behind, so wait 2 seconds after sending.
We save the id of the last message. This is a **cursor**. On the next run we ask only for messages after it.
This is an **incremental load**. Run this cell many times.

> **Run: many times.** This is the step that **saves** orders. Run it after orders are placed, and again any time you want the newest ones.

In [ ]:
time.sleep(2)                      # brokers can be slightly behind
if "raw_events" not in globals():  # first run only
    raw_events, last_id = [], None
new = poll(CLASS_TOPIC, last_id or "all")   # only messages after the cursor
raw_events += new
if new:
    last_id = new[-1]["id"]        # move the cursor
print("New messages this run:", len(new), "| total stored:", len(raw_events))

### 5c. Streaming consumer (presenter demo)
Instead of asking again and again, we keep the connection open. Messages are printed as they arrive.
The presenter will show this live on screen. **Participants: skip this step.** While it listens, Colab cannot run any other cell, so its code is commented out on purpose.

> **Run: presenter only.** Participants go straight to 5d. Run 5b once more to collect any new orders.

In [ ]:
# PRESENTER ONLY. Participants: do not run this cell.
# It keeps Colab busy while it listens, so no other cell can run until it stops.
# To demo it, remove the leading '# ' from the lines below.
#
# STREAM_SECONDS = 30  # @param {type:"integer"}
# def show(event):
#     print("LIVE:", event["message"])
# print("Listening for", STREAM_SECONDS, "seconds...")
# stream(CLASS_TOPIC, STREAM_SECONDS, show)
# print("Closed.")

### 5d. Clean
Real data is messy. A ntfy `message` is a **string** of JSON, so we convert it. Bad messages are skipped and counted. Duplicates are removed using `order_id`.

> **Run: once, after 5b.**

In [ ]:
assert "raw_events" in globals(), "Run step 5b first. It collects the orders. Then run this cell again."

rows, skipped = [], 0
for event in raw_events:
    try:
        o = json.loads(event["message"])   # the message is a string of JSON text
        order = {"order_id": o["order_id"], "customer": o["customer"],
                 "product_id": o["product_id"], "qty": o["qty"], "ts": o["ts"]}
        # keep it only if every field has the right type and qty is above 0
        valid = (isinstance(order["order_id"], str) and isinstance(order["customer"], str)
                 and type(order["product_id"]) is int and type(order["qty"]) is int
                 and order["qty"] > 0)
    except (ValueError, KeyError, TypeError):   # not JSON, or a field is missing
        valid = False
    if valid:
        rows.append(order)
    else:
        skipped += 1

all_orders = pd.DataFrame(rows, columns=["order_id", "customer", "product_id", "qty", "ts"])
orders = all_orders.drop_duplicates("order_id").reset_index(drop=True)   # same order_id = duplicate

print("Rows before dedupe:", len(all_orders))
print("Rows after dedupe: ", len(orders))
print("Skipped (not valid orders):", skipped)
orders.head()

### 5e. Enrich and answer
Join the orders with the `products` table from section 1. Then revenue = price × qty.

> **Run: once, after 5d.** It also needs the `products` table from section 1.

In [ ]:
assert "orders" in globals(), "Run step 5d first (clean)."
assert "products" in globals(), "Run the products cell in section 1 first."

# 1. Join each order with its product: match product_id with the product's id
enriched = orders.merge(products, left_on="product_id", right_on="id", how="left")

# 2. Revenue of each order = price x quantity
enriched["revenue"] = enriched["price"] * enriched["qty"]

# 3. Add up the revenue per product, biggest first, and keep the top 5
top = (enriched.groupby("title", as_index=False)["revenue"].sum()
       .sort_values("revenue", ascending=False)
       .head(5)
       .reset_index(drop=True))

print("Best-seller tonight:", top.iloc[0]["title"] if len(top) else "no orders yet")
top

### 5f. Optional: export to CSV
A **CSV** file is a plain table you can open in Excel or Google Sheets. This step is optional. It saves the clean tables you just built.

> **Run: once.** Optional.

In [ ]:
# Optional: save the clean tables as CSV files
products.to_csv("products.csv", index=False)
enriched.to_csv("orders_enriched.csv", index=False)
top.to_csv("top_products.csv", index=False)
print("Saved: products.csv, orders_enriched.csv, top_products.csv")
try:
    from google.colab import files        # only exists inside Colab
    files.download("top_products.csv")    # sends the file to your computer
except ImportError:
    print("Not in Colab: the files are next to this notebook.")

## 6. Go further
- **Postman**: click-to-try any API, no code.
- **Make / n8n**: build webhook receivers with boxes and arrows.
- **dlt**: Python library that loads API data into a database.
- **Airbyte**: ready-made connectors for hundreds of sources.
- **Kafka**: big event broker. A ntfy topic ≈ a Kafka topic. Our cursor ≈ a consumer offset.
- **Debezium / CDC**: turns database changes into events.
- **Airflow**: schedules and monitors pipelines.